# SynBioCrow M2.4.11.5 — Embedded ReactionTree export repair

**Goal:** Extract genuine native reaction nodes from the live `reaction_tree` object inside each AiZynthFinder route dictionary.

**Context:** M2.4.11.4 completed 15 and 16 routes but serialized object representations rather than reaction trees.

**Constraints:** Keep the proven Python 3.11, AiZynthFinder 4.4.1, NumPy/PyArrow pins, USPTO expansion policy, ZINC stock and two test targets. Do not claim chemistry certification.

**Done when:** Both targets have native reaction nodes with concrete SMILES and the output ZIP matches this notebook's name.


In [ ]:
from pathlib import Path
import subprocess, shutil, tarfile, urllib.request, json
ROOT=Path('/content/synbiocrow_aizynth'); ROOT.mkdir(parents=True,exist_ok=True)
OUT=ROOT/'outputs'; OUT.mkdir(exist_ok=True)
MAMBA=ROOT/'bin/micromamba'; ENV=ROOT/'env_py311'; PY=ENV/'bin/python'
def run(cmd,name,timeout=1800):
 print('RUN:', ' '.join(map(str,cmd)),flush=True)
 try:
  p=subprocess.run([str(x) for x in cmd],capture_output=True,text=True,timeout=timeout)
  (OUT/name).write_text(p.stdout+'\nSTDERR\n'+p.stderr)
  print('RC',p.returncode,'TAIL',(p.stdout+p.stderr)[-1800:],flush=True)
  return p.returncode
 except Exception as e:
  (OUT/name).write_text(repr(e)); print('FAILED',repr(e)); return -1
if not MAMBA.exists():
 MAMBA.parent.mkdir(exist_ok=True)
 with urllib.request.urlopen('https://micro.mamba.pm/api/micromamba/linux-64/latest',timeout=120) as response:
  with tarfile.open(fileobj=response,mode='r|bz2') as archive:
   for member in archive:
    if member.name=='bin/micromamba':
     with archive.extractfile(member) as inp,open(MAMBA,'wb') as out:shutil.copyfileobj(inp,out)
     break
 MAMBA.chmod(0o755)
if not PY.exists():
 if run([MAMBA,'create','-y','-p',ENV,'-c','conda-forge','python=3.11','pip'],'micromamba_create.log'):raise RuntimeError('Python 3.11 setup failed')
if run([PY,'-m','pip','install','--disable-pip-version-check','aizynthfinder==4.4.1','numpy==1.26.4','pyarrow==17.0.0'],'pip_install.log',1800):raise RuntimeError('Pinned installation failed')
if run([PY,'-c','import numpy,pyarrow; from aizynthfinder.aizynthfinder import AiZynthFinder; print("IMPORT_PASS",numpy.__version__,pyarrow.__version__)'],'import_postflight.log',180):raise RuntimeError('Native import failed')
DATA=ROOT/'public_data'; DATA.mkdir(exist_ok=True)
if not (DATA/'config.yml').exists():
 rc=run([ENV/'bin/download_public_data',DATA],'public_data_download.log',3600)
 if rc:
  print('Public model download failed; inspect public_data_download.log',flush=True)
  raise RuntimeError('AiZynthFinder official public model download failed; cannot run native search')
print('Config present:',(DATA/'config.yml').exists())

In [ ]:
NATIVE_RUNNER = "import json,traceback\nfrom pathlib import Path\nfrom aizynthfinder.aizynthfinder import AiZynthFinder\nROOT=Path('/content/synbiocrow_aizynth');OUT=ROOT/'outputs';OUT.mkdir(exist_ok=True)\nCONFIG=ROOT/'public_data/config.yml'\ntargets={'3_hydroxypropionic_acid':'O=C(O)CCO','butanal':'CCCC=O'}\nsummary={'engine':'AiZynthFinder 4.4.1','mode':'native chemical retrosynthesis','targets':{},'model_config_exists':CONFIG.exists()}\ndef tree_dict(route):\n # RouteCollection entries are dictionaries whose reaction_tree is a live ReactionTree.\n if isinstance(route,dict):\n  tree=route.get('reaction_tree')\n  if tree is None:raise ValueError('Native route dictionary lacks reaction_tree')\n  if hasattr(tree,'to_dict'):return tree.to_dict()\n  if isinstance(tree,dict):return tree\n  raise TypeError('Unexpected reaction_tree type '+str(type(tree)))\n if hasattr(route,'reaction_tree'):\n  tree=route.reaction_tree\n  if hasattr(tree,'to_dict'):return tree.to_dict()\n  if isinstance(tree,dict):return tree\n if hasattr(route,'to_dict'):return route.to_dict()\n raise TypeError('Unrecognized native route type '+str(type(route)))\ndef collect(node,records,path='root'):\n if isinstance(node,dict):\n  kind=str(node.get('type','')).lower()\n  if kind=='reaction':\n   records.append({'smiles':node.get('smiles'),'metadata':node.get('metadata',{}),'type':node.get('type'),'tree_path':path})\n  for k,v in node.items():\n   if isinstance(v,(dict,list)):collect(v,records,path+'/'+str(k))\n elif isinstance(node,list):\n  for i,v in enumerate(node):collect(v,records,path+'/'+str(i))\nif not CONFIG.exists():summary['error']='Official model configuration missing'\nelse:\n for name,smi in targets.items():\n  rec={'target_smiles':smi,'status':'STARTED','routes':[],'native_reaction_nodes':[]}\n  try:\n   finder=AiZynthFinder(configfile=str(CONFIG))\n   available=list(finder.expansion_policy.items)\n   rec['available_expansion_policies']=available\n   if not available:raise RuntimeError('No expansion policy loaded')\n   preferred=next((x for x in available if 'uspto' in x.lower()),available[0]);finder.expansion_policy.select(preferred)\n   if not list(finder.expansion_policy.selection):raise RuntimeError('Expansion policy selection did not persist')\n   rec['selected_expansion_policy']=preferred\n   stocks=list(finder.stock.items);rec['available_stocks']=stocks\n   if stocks:finder.stock.select(stocks[0]);rec['selected_stock']=stocks[0]\n   finder.target_smiles=smi\n   finder.config.search.time_limit=90;finder.config.search.iteration_limit=150\n   finder.prepare_tree();finder.tree_search();finder.build_routes()\n   rec['route_count']=len(finder.routes)\n   for idx,route in enumerate(finder.routes):\n    try:\n     d=tree_dict(route)\n     rec['routes'].append({'route_index':idx,'tree':d,'route_metadata':route.get('route_metadata',{}) if isinstance(route,dict) else {}})\n     collect(d,rec['native_reaction_nodes'],'route_'+str(idx))\n    except Exception as e:\n     rec.setdefault('route_errors',[]).append({'route_index':idx,'error':repr(e),'type':str(type(route))})\n   rec['native_reaction_nodes_count']=len(rec['native_reaction_nodes'])\n   rec['unique_reaction_smiles']=sorted({str(r.get('smiles')) for r in rec['native_reaction_nodes'] if r.get('smiles')})\n   rec['unique_reaction_count']=len(rec['unique_reaction_smiles'])\n   rec['reaction_node_examples']=rec['native_reaction_nodes'][:3]\n   rec['status']='NATIVE_REACTION_NODES_EXPORTED' if rec['native_reaction_nodes'] else 'SEARCH_COMPLETED_NO_EXPORTED_REACTIONS'\n  except Exception as e:\n   rec['status']='ERROR';rec['error']=repr(e);rec['traceback']=traceback.format_exc()\n  summary['targets'][name]={k:v for k,v in rec.items() if k not in ('routes','native_reaction_nodes')}\n  (OUT/(name+'_native_routes.json')).write_text(json.dumps(rec,indent=2,default=str))\n(OUT/'native_summary.json').write_text(json.dumps(summary,indent=2,default=str))\nprint(json.dumps(summary,indent=2,default=str))\n"

from pathlib import Path
import subprocess
ROOT=Path('/content/synbiocrow_aizynth');OUT=ROOT/'outputs';OUT.mkdir(exist_ok=True)
script=ROOT/'native_probe.py'
script.write_text(NATIVE_RUNNER)
p=subprocess.run([str(ROOT/'env_py311/bin/python'),str(script)],capture_output=True,text=True,timeout=1200)
(OUT/'native_probe_stdout.log').write_text(p.stdout)
(OUT/'native_probe_stderr.log').write_text(p.stderr)
print('Native runner exit:',p.returncode,'stdout:',p.stdout[-3500:],'stderr:',p.stderr[-2500:])

In [ ]:
from pathlib import Path
import zipfile
ROOT=Path('/content/synbiocrow_aizynth');OUT=ROOT/'outputs'
DEST=Path('/content/SynBioCrow_M2_4_11_5_AIZYNTHFINDER_EMBEDDED_TREE_EXPORT_REPAIR_ALL_OUTPUTS.zip')
with zipfile.ZipFile(DEST,'w',compression=zipfile.ZIP_DEFLATED) as z:
 for p in sorted(OUT.rglob('*')):
  if p.is_file():z.write(p,'outputs/'+str(p.relative_to(OUT)))
print('Output:',DEST,'bytes:',DEST.stat().st_size)
try:
 from google.colab import files
 files.download(str(DEST))
except Exception as e:print('Automatic download unavailable:',repr(e))

## Rescue download
Use only if automatic download was blocked.

In [ ]:
from google.colab import files
files.download('/content/SynBioCrow_M2_4_11_5_AIZYNTHFINDER_EMBEDDED_TREE_EXPORT_REPAIR_ALL_OUTPUTS.zip')